In [13]:
import yaml
import numpy as np
import matplotlib.pyplot as plt

from foilpolars.shapes import load_all_shapes, load_raw_shapes, plot_shapes
from foilpolars.grassmann import (
    compute_grassmann,
    check_reconstruction,
    compute_pga_basis,
    compute_pga_embedding,
    perturb_grassmann,
    plot_pga_embedding,
    plot_perturbed_shapes,
    plot_pga_pairs,
    plot_grassmann_perturbed,
)

# Config drives the same run as `foilpolars grassmann`, so this notebook
# reproduces its steps one at a time with intermediate plots
with open("../configs/sweep_config.yaml") as f:
    config = yaml.safe_load(f)

In [14]:
# Step 1: load the baseline foils, repaneled to a common point count
shapes = load_all_shapes(config)
n_points = {desig: coords.shape[0] for desig, coords in shapes.items()}
print(f"{len(shapes)} baseline foils, points per foil: {n_points}")
plot_shapes(shapes)

19 baseline foils, points per foil: {'sg6043': 199, 'naca633218': 199, 'naca4415': 199, 'naca16018': 199, 'naca642415': 199, 'naca654421': 199, 'naca66-018': 199, 's1223': 199, 'e420': 199, 'e421': 199, 'clarky': 199, 'naca63209': 199, 'naca64210': 199, 'naca65210': 199, 'naca0015': 199, 'naca0018': 199, 'naca0021': 199, 'fx74cl5140': 199, 'fx74cl6140': 199}


In [15]:
# Step 2: standardize each foil independently onto the Grassmann
# manifold (translation removed via b, rotation/scale removed via M)
raw_shapes = load_raw_shapes(config)
results = compute_grassmann(raw_shapes)
check_reconstruction(raw_shapes, results)

example = "naca4415"
X_gr, M, b = (
    results[example]["X_gr"], results[example]["M"], results[example]["b"],
)
print(f"{example}: X_gr {X_gr.shape}, M {M.shape}, b {b.shape}")

sg6043: max reconstruction error = 4.44e-16
naca633218: max reconstruction error = 2.22e-16
naca4415: max reconstruction error = 1.73e-16
naca16018: max reconstruction error = 2.22e-16
naca642415: max reconstruction error = 4.44e-16
naca654421: max reconstruction error = 1.28e-16
naca66-018: max reconstruction error = 1.73e-16
s1223: max reconstruction error = 8.88e-16
e420: max reconstruction error = 3.33e-16
e421: max reconstruction error = 3.33e-16
clarky: max reconstruction error = 3.33e-16
naca63209: max reconstruction error = 2.22e-16
naca64210: max reconstruction error = 2.22e-16
naca65210: max reconstruction error = 2.22e-16
naca0015: max reconstruction error = 1.11e-16
naca0018: max reconstruction error = 2.34e-16
naca0021: max reconstruction error = 1.11e-16
fx74cl5140: max reconstruction error = 2.22e-16
fx74cl6140: max reconstruction error = 2.74e-16
naca4415: X_gr (97, 2), M (2, 2), b (2,)


In [16]:
# Compare one foil's raw physical shape to its standardized Grassmann
# shape: same outline, but centered and scaled to unit spread
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(raw_shapes[example][:, 0], raw_shapes[example][:, 1])
axes[0].set_title(f"{example}: raw physical (x/c, y/c)")
axes[0].set_box_aspect(1)
axes[1].plot(X_gr[:, 0], X_gr[:, 1], color="tab:orange")
axes[1].set_title(f"{example}: X_gr (Grassmann shape)")
axes[1].set_box_aspect(1)
plt.tight_layout()
plt.show()

In [17]:
# Step 3-4: batch-align all foils (Procrustes), then compute the
# Karcher mean and the 4-mode PGA basis over the aligned shapes
repaneled_shapes = load_all_shapes(config)
basis = compute_pga_basis(repaneled_shapes, n_coord=4)
print(f"mu (Karcher mean): {basis['mu'].shape}")
print(f"Vh (PGA basis): {basis['Vh'].shape}")
print(f"t (PGA coords per foil): {basis['t'].shape}")

Grassmann manifold
Karcher mean convergence:
||V||_F = 0.2751954741923166
||V||_F = 0.002475796584451241
||V||_F = 6.996011516363996e-05
||V||_F = 2.0460506167805288e-06
||V||_F = 5.989389270574503e-08
||V||_F = 1.754388535815161e-09
mu (Karcher mean): (199, 2)
Vh (PGA basis): (4, 398)
t (PGA coords per foil): (19, 4)


In [18]:
# Reconstruct the Karcher mean into physical coordinates using the
# mean affine transform, then compare it against the baseline foils
m_mean = np.mean(basis["M"], axis=0)
b_mean = np.mean(basis["b"], axis=0)
karcher_phys = basis["mu"] @ m_mean + b_mean

fig, ax = plt.subplots(figsize=(8, 4))
for desig, coords in repaneled_shapes.items():
    ax.plot(coords[:, 0], coords[:, 1], color="grey", linewidth=0.5)
ax.plot(
    karcher_phys[:, 0], karcher_phys[:, 1], color="k", linewidth=2,
    label="Karcher mean",
)
ax.set_box_aspect(1)
ax.set_title("Step 4: Karcher mean vs. baseline foils")
ax.legend()
plt.show()

In [19]:
# Step 4b: project each baseline foil onto its first 2 PGA coordinates,
# collapsing each full shape to a single point in "shape space"
foil_ids, t2 = compute_pga_embedding(repaneled_shapes)
plot_pga_embedding(t2, title="Step 4: baseline foils in 2D PGA space")

Grassmann manifold
Karcher mean convergence:
||V||_F = 0.2751954741923166
||V||_F = 0.002475796584451241
||V||_F = 6.996011516363996e-05
||V||_F = 2.0460506167805288e-06
||V||_F = 5.989389270574503e-08
||V||_F = 1.754388535815161e-09


In [20]:
# Step 5-6: sample new shapes by drawing the 4 PGA coords + thickness
# ratio uniformly within the baseline foils' range, exp-mapping each
# draw back to a physical shape, and rejecting self-intersecting draws
grassmann_config = config.get("grassmann", {})
perturbed = perturb_grassmann(
    basis,
    n_perturb=grassmann_config.get("n_perturb", 20),
    seed=grassmann_config.get("seed"),
)
print(f"Sampled {len(perturbed['phys'])} perturbed shapes")
print(
    f"phys {perturbed['phys'].shape}, coef {perturbed['coef'].shape}, "
    f"thickness_ratio {perturbed['thickness_ratio'].shape}"
)

Sampled 1000 perturbed shapes
phys (1000, 199, 2), coef (1000, 4), thickness_ratio (1000,)


In [21]:
# Baseline vs. perturbed shapes, in both physical and Grassmann space
plot_perturbed_shapes(repaneled_shapes, basis, perturbed)

In [22]:
# Corner plot of the 5 sampled parameters: 4 PGA coords + thickness
# ratio, baseline foils overlaid as stars
plot_pga_pairs(basis, perturbed)

In [23]:
# All perturbed samples overlaid on the Grassmann manifold
# representation, alongside the baseline foils
plot_grassmann_perturbed(basis, perturbed)

In [24]:
# Step 7: these perturbed shapes (here, 1000 as in sweep_config.yaml)
# are what sweep.py runs through XFoil/NeuralFoil across alpha/Re/n_crit
print(f"Total shapes swept downstream: {len(perturbed['phys'])}")

Total shapes swept downstream: 1000
